In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_community.document_loaders import UnstructuredPDFLoader
from langchain_community.embeddings import HuggingFaceEmbeddings

In [ ]:
CHUNK_SIZE = 1000
CHUNK_OVERLAP = 250
pdf_path = '../Database/Pdfs/Digital_Personal_Data_Protection_Act_2023.pdf'
loader = UnstructuredPDFLoader(file_path=pdf_path,
                            mode='elements',
                            chunking_strategy="by_title",
                            max_characters=1000,
                            combine_text_under_n_chars=500,
                            unstructured_kwargs={'output_type':'html'})


In [3]:
embedding_model = HuggingFaceEmbeddings(model_name = "BAAI/bge-base-en-v1.5")

C:\Users\naray\AppData\Local\Temp\ipykernel_14568\3837422460.py:1: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embedding_model = HuggingFaceEmbeddings(model_name = "BAAI/bge-base-en-v1.5")


What we need basically for now are: 

metadata : title,filename,page_number,category(title or narrative text or ListItem etc)

content : page_content

In [4]:
docs = loader.load()
categories = ['title','NarrativeText','ListItem']
filtered = [doc for doc in docs if doc.metadata.get('category') in categories]
spliter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, 
                                        chunk_overlap=CHUNK_OVERLAP,
                                        separators=["\n\n", "\n", " ", ""])
new_docs = spliter.split_documents(documents=filtered)

In [5]:
len(docs),len(filtered),len(new_docs)

(91, 0, 0)

In [ ]:
# spliter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
# x = spliter.split_text("hellow world we need your help in this. \n\n What are your thoughts on this")
len(docs),len(filtered),len(new_docs)
docs_text = [doc.page_content for doc in docs]
db = Chroma(collection_name='vector',
            embedding_function=embedding_model)
# embeddings = embedding_model.embed_documents(docs_text)

C:\Users\naray\AppData\Local\Temp\ipykernel_14568\370639552.py:5: LangChainDeprecationWarning: The class `Chroma` was deprecated in LangChain 0.2.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-chroma package and should be used instead. To use it run `pip install -U :class:`~langchain-chroma` and import as `from :class:`~langchain_chroma import Chroma``.
  db = Chroma(collection_name='vector',embedding_function=embedding_model)


In [7]:
from langchain_community.vectorstores.utils import filter_complex_metadata
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers import EnsembleRetriever
simplified_docs = filter_complex_metadata(documents=docs)
db.add_documents(documents=simplified_docs)  
bm25 = BM25Retriever.from_documents(documents=simplified_docs)
bm25.k = 3
retrievers = [bm25,
            db.as_retriever(search_kwargs={'k': 3})] 
ranked_retriever = EnsembleRetriever(
    retrievers=retrievers,
    weights=[0.5,0.5]
)

In [8]:
_query = "What is Digital Data Protection Act ?"
results1 = db.search(query=_query,search_type='similarity_score_threshold',score_threshold=0.7)
results2 = db.search(query=_query,search_type='similarity')
results3 = db.search(query=_query,search_type='mmr')


In [47]:
content = []
for result in results1:
    content.append(result.page_content)
content

['THE DIGITAL PERSONAL DATA PROTECTION ACT, 2023(NO. 22 OF 2023)[11th August, 2023.]An Act to provide for the processing of digital personal data in a manner thatrecognises both the right of individuals to protect their personal data and theneed to process such personal data for lawful purposes and for mattersconnected therewith or incidental thereto.BE it enacted by Parliament in the Seventy-fourth Year of the Republic of India asfollows:––CHAPTER IPRELIMINARY1. (1) This Act may be called the Digital Personal Data Protection Act, 2023.(2) It shall come into force on such date as the Central Government may, by notificationin the Official Gazette, appoint and different dates may be appointed for different provisionsof this Act and any reference in any such provision to the commencement of this Act shallbe construed as a reference to the coming into force of that provision.Short title andcommencement.']

In [41]:
content = []
for result in results3:
    content.append(result.page_content)
content

['THE DIGITAL PERSONAL DATA PROTECTION ACT, 2023(NO. 22 OF 2023)[11th August, 2023.]An Act to provide for the processing of digital personal data in a manner thatrecognises both the right of individuals to protect their personal data and theneed to process such personal data for lawful purposes and for mattersconnected therewith or incidental thereto.BE it enacted by Parliament in the Seventy-fourth Year of the Republic of India asfollows:––CHAPTER IPRELIMINARY1. (1) This Act may be called the Digital Personal Data Protection Act, 2023.(2) It shall come into force on such date as the Central Government may, by notificationin the Official Gazette, appoint and different dates may be appointed for different provisionsof this Act and any reference in any such provision to the commencement of this Act shallbe construed as a reference to the coming into force of that provision.Short title andcommencement.',
 'this Act;\n\n(x) “processing” in relation to personal data, means a wholly or partl

In [9]:
result_hybrid_docs = ranked_retriever.invoke(_query)

In [10]:
result_hybrid_docs

[Document(metadata={'source': '../Database/Pdfs/Digital_Personal_Data_Protection_Act_2023.pdf', 'file_directory': '../Database/Pdfs', 'filename': 'Digital_Personal_Data_Protection_Act_2023.pdf', 'last_modified': '2025-09-24T17:54:20', 'page_number': 1, 'orig_elements': 'eJxdVMFO4zAQ/ZVRTiC1oU3LLnDLQhcqsW1UckEsilxn0lhK7Mh2gC7i33fGDaLiEmXGb2bee57k6T3CBlvUvlBldAVR9WO2xZmYj2dS4ngup+VYXM7LsZQTkVwIMSlxEo0gatGLUnhBNe+RNMaWSguPLsSN2JveFzWqXe0pczFP4gkVDflXVfqa0ueX5yHdGaU9Vz49TThBj+cRHN5DKUcD+HvMWAojt3ceW1aQqTdsHjohMfqgg0o1WJTKovTG7hkQx2c3xHwrHJ5lZeWiAaVFi3x+o3bKi6bI0Dqj6YXRRWaNpx7K6CKVvkgmySzuyioKspwvWlOqSmEwkQ7Px5PLcTLPpz+vzudXSfCsEzssdN9u0RJqyuw8vrFBUX63gJvl7TJP7yFbbB7WK3q5SfMUss06X1zny/UK0ut8BDz4ZLWOIUlg/TuEp0/Tqa8h7Xe98wdE/JxqIJ7gDXTWvKgSoTIWfI0cS3RO6R2YCsqDWugGtcC3CkqDgFZojVwjPNlndlo5dLA1NIv7WL5dbqF0qWhCLxo3zGOnGKPst75Cl5zXiOUAZSrgell/QzLbRrxWPVHrbWd4NFdzvhXeE1ga4ic9hpYWXxURo1OlJcnVLCrkvYl/LUB5oBsO6O0eMmEbJXjvWSqrecAXivbjyvSW+jyisKyNjzbY9dtGSY6XpJVkuMo0jXl1V3/7ZDKdHZ7Xd2mWLzawzDaL++Wf5SrdP